In [15]:
import os
import sys
import time
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# SHARED CONFIGURATION PARAMETERS
# =============================================================================

# -- File & Path Configuration --
HDF_FILE_PATH = '../../data/raw/all_hourly_data.h5'
FEATURE_CLASSIFICATION_PATH = '../../data/processed/eda_results_corrected/feature_classification.csv'
OUTPUT_DIR = 'phase_1_outputs'

# -- Experiment Configuration --
DRY_RUN = False  # Set to False for full dataset
DRY_RUN_PATIENTS = 2000  # Number of patients for dry run
USE_CACHED_PREPROCESSING = True  # Use cache if available

# -- Feature Engineering Options --
CALCULATE_TRENDS = True  # Include trend calculations
WINDOW_SIZE = 24  # Hours of data for feature extraction
GAP_TIME = 6      # Hours of gap before prediction

# -- Target and Reproducibility --
TARGET_VARIABLE = 'mort_hosp'
SEED = 42

# -- Hyperparameter Tuning Configuration --
N_OPTUNA_TRIALS = 15  # Number of trials for hyperparameter search
OPTUNA_TIMEOUT = 1800 # Timeout in seconds

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [16]:
print("=" * 70)
print("PHASE 1: CLEAN DATA PREPROCESSING")
print("=" * 70)

preprocessing_start = time.time()

# Import and run preprocessing
try:
    # Import the clean preprocessing script as a module
    import importlib.util
    spec = importlib.util.spec_from_file_location("data_preprocessing_clean", "data_preprocessing_clean.py")
    if spec is None:
        raise ImportError("Could not load data_preprocessing_clean.py")
    
    preprocessing_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for data_preprocessing_clean.py")
    
    # Execute the preprocessing script
    print(f"Starting clean data preprocessing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    spec.loader.exec_module(preprocessing_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'HDF_FILE_PATH': HDF_FILE_PATH,
        'FEATURE_CLASSIFICATION_PATH': FEATURE_CLASSIFICATION_PATH,
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'USE_CACHED_PREPROCESSING': USE_CACHED_PREPROCESSING,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED
    }
    
    # Call the main function explicitly with configuration
    preprocessing_module.main(config)
    
    preprocessing_time = time.time() - preprocessing_start
    print(f"\n✓ Clean data preprocessing completed in {preprocessing_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during preprocessing: {e}")
    raise


2025-07-01 22:58:38,615 [INFO] - Loading data...
2025-07-01 22:58:38,680 [INFO] - NumExpr defaulting to 8 threads.


PHASE 1: CLEAN DATA PREPROCESSING
Starting clean data preprocessing at 2025-07-01 22:58:38


2025-07-01 22:58:43,609 [INFO] - Age filtered: 32550 patients, 2080959 records
2025-07-01 22:58:50,594 [INFO] - Time window filtered: 22591 patients, (542184, 104) time-series
2025-07-01 22:58:50,622 [INFO] - Splits: train=14825, val=2118, test=5648 subjects
2025-07-01 22:59:34,400 [INFO] - Demographics processed: (22591, 4)
2025-07-01 22:59:34,677 [INFO] - Engineering features for 6 categories...
2025-07-01 22:59:34,678 [INFO] -   - Sparse Dynamic: 12 features
2025-07-01 22:59:34,678 [INFO] -   - Stable Index: 6 features
2025-07-01 22:59:34,679 [INFO] -   - Event-Driven: 26 features
2025-07-01 22:59:34,679 [INFO] -   - Labile Lab: 46 features
2025-07-01 22:59:34,679 [INFO] -   - High-Frequency Physiological: 12 features
2025-07-01 22:59:34,680 [INFO] -   - Static: 2 features
2025-07-01 23:04:48,312 [INFO] - Features engineered: (14825, 388)
2025-07-01 23:04:48,437 [INFO] - Engineering features for 6 categories...
2025-07-01 23:04:48,438 [INFO] -   - Sparse Dynamic: 12 features
2025-07


✓ Clean data preprocessing completed in 8.92 minutes


## 3. XGBoost Model Training & Evaluation

Run XGBoost analysis with hyperparameter optimization using the preprocessed data:


In [17]:
print("=" * 70)
print("PHASE 2: XGBOOST MODEL TRAINING & EVALUATION")
print("=" * 70)

xgboost_start = time.time()

# Initialize variables for results capture
auroc_results = None
auprc_results = None
final_model = None
xgboost_results = None

try:
    # Import and run XGBoost analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("xgboost_analysis", "xgboost_analysis.py")
    if spec is None:
        raise ImportError("Could not load xgboost_analysis.py")
    
    xgboost_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for xgboost_analysis.py")
    
    print(f"Starting XGBoost analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the XGBoost analysis module and call main function
    spec.loader.exec_module(xgboost_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function explicitly with configuration
    xgboost_module.main(config)
    
    # Load the results that were saved by the XGBoost script
    results_path = os.path.join(OUTPUT_DIR, 'results_xgboost_baseline.pkl')
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            xgboost_results = pickle.load(f)
        
        # Extract key metrics for display
        auroc_results = {
            'point_estimate': xgboost_results['test_auroc'],
            'ci_lower': xgboost_results['test_auroc_ci_lower'],
            'ci_upper': xgboost_results['test_auroc_ci_upper']
        }
        
        auprc_results = {
            'point_estimate': xgboost_results['test_auprc'], 
            'ci_lower': xgboost_results['test_auprc_ci_lower'],
            'ci_upper': xgboost_results['test_auprc_ci_upper']
        }
        
        print(f"\n✅ XGBoost Results Successfully Captured:")
        print(f"   AUROC: {auroc_results['point_estimate']:.4f} "
              f"(95% CI: {auroc_results['ci_lower']:.4f}-{auroc_results['ci_upper']:.4f})")
        print(f"   AUPRC: {auprc_results['point_estimate']:.4f} "
              f"(95% CI: {auprc_results['ci_lower']:.4f}-{auprc_results['ci_upper']:.4f})")
    else:
        print("⚠️  Results file not found - metrics may not be available in summary")
    
    xgboost_time = time.time() - xgboost_start
    print(f"\n✓ XGBoost analysis completed in {xgboost_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during XGBoost analysis: {e}")
    print(f"   This may be due to XGBoost library installation issues")
    print(f"   Preprocessing completed successfully - data is ready for manual analysis")
    xgboost_time = time.time() - xgboost_start
    print(f"   Attempted runtime: {xgboost_time/60:.2f} minutes")
    # Don't raise - allow pipeline to continue and show preprocessing results


2025-07-01 23:07:33,671 [INFO] - Loading preprocessed data with prefix: preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42


2025-07-01 23:07:33,729 [INFO] - Data shapes: X_train=(14825, 392), X_val=(2118, 392), X_test=(5648, 392)
2025-07-01 23:07:33,730 [INFO] - Performing data validation for XGBoost...
2025-07-01 23:07:33,750 [INFO] - ✓ All 392 columns are numeric


PHASE 2: XGBOOST MODEL TRAINING & EVALUATION
Starting XGBoost analysis at 2025-07-01 23:07:33


2025-07-01 23:07:34,287 [INFO] - ✓ Preserved 1614463 NaN values for XGBoost missingness learning
2025-07-01 23:07:34,288 [INFO] - ✓ Data type conversion completed for XGBoost compatibility
2025-07-01 23:07:34,288 [INFO] - Starting Optuna search with 15 trials...
[I 2025-07-01 23:07:34,290] A new study created in memory with name: no-name-541b949a-e2ec-46c5-8b08-165fba2a6762
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  UserWarning,
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  UserWarning,
[I 2025-07-01 23:07:42,204] Trial 0 finished with value: 0.8561


Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      5077
           1       0.54      0.58      0.56       571

    accuracy                           0.91      5648
   macro avg       0.74      0.76      0.75      5648
weighted avg       0.91      0.91      0.91      5648


Confusion Matrix:
[[4789  288]
 [ 238  333]]

✅ XGBoost Results Successfully Captured:
   AUROC: 0.9095 (95% CI: 0.8984-0.9199)
   AUPRC: 0.6102 (95% CI: 0.5736-0.6452)

✓ XGBoost analysis completed in 10.71 minutes


## 4. Elastic Net Model Training & Evaluation

Run Elastic Net analysis with hyperparameter optimization using the preprocessed data. Note that Elastic Net requires imputation of missing values since it cannot handle NaN values like XGBoost can:


In [18]:
print("=" * 70)
print("PHASE 3: ELASTIC NET MODEL TRAINING & EVALUATION")
print("=" * 70)

elastic_net_start = time.time()

try:
    # Import and run Elastic Net analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("elastic_net_analysis", "elastic_net_analysis.py")
    if spec is None:
        raise ImportError("Could not load elastic_net_analysis.py")
    
    elastic_net_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for elastic_net_analysis.py")
    
    print(f"Starting Elastic Net analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the Elastic Net analysis module
    spec.loader.exec_module(elastic_net_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function
    elastic_net_module.main(config)
    
    elastic_net_time = time.time() - elastic_net_start
    print(f"\\n✓ Elastic Net analysis completed in {elastic_net_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during Elastic Net analysis: {e}")
    elastic_net_time = time.time() - elastic_net_start
    print(f"   Attempted runtime: {elastic_net_time/60:.2f} minutes")


2025-07-01 23:27:22,458 [INFO] - Loading preprocessed data with prefix: preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42
2025-07-01 23:27:22,552 [INFO] - Data shapes: X_train=(14825, 392), X_val=(2118, 392), X_test=(5648, 392)
2025-07-01 23:27:22,553 [INFO] - Performing data validation and imputation for Elastic Net...
2025-07-01 23:27:22,565 [INFO] - ✓ All 392 columns are numeric


PHASE 3: ELASTIC NET MODEL TRAINING & EVALUATION
Starting Elastic Net analysis at 2025-07-01 23:27:22


2025-07-01 23:27:23,058 [INFO] - Total missing values before imputation: 2039274
2025-07-01 23:27:23,543 [INFO] - ✓ Successfully imputed 2039274 missing values using median strategy
2025-07-01 23:27:23,543 [INFO] - ✓ Data prepared for Elastic Net (no missing values)
2025-07-01 23:27:23,544 [INFO] - Starting Optuna search with 15 trials...
[I 2025-07-01 23:27:23,544] A new study created in memory with name: no-name-a0523ff2-0e8c-4bfd-8cf2-f822c64fe3f4
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/sklearn/linear_model/_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
[I 2025-07-01 23:28:55,574] Trial 0 finished with value: 0.689630291368884 and parameters: {'C': 0.0040624502969366875, 'l1_ratio': 0.737104109623903, 'max_iter': 2000}. Best is trial 0 with value: 0.689630291368884.
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/sklearn/linear_model/_sag.py:354: Converge